In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
import os
import json
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib


TARGET_WELLS = ['15/9-F-12', '15/9-F-14']
MODELS_DIR = '../outputs/models'

print(f'\t Loading data ')
df = pd.read_parquet('../data/processed/cleaned.parquet')
df['DATEPRD'] = pd.to_datetime(df['DATEPRD'])
df = df.set_index('DATEPRD').sort_index()
df['OIL_RATE_NORM'] = df.groupby('NPD_WELL_BORE_NAME')['OIL_RATE_NORM'].transform(
    lambda x :x.clip(upper= x.quantile(0.99))
)
print(f'Loaded  {len(df):,} rows')

print(f'Resampling to monthly')
AGG_RULES = {
    'OIL_RATE_NORM': 'mean',
    'BORE_OIL_VAL' : 'mean',
    'AVG_WHP_P':      'mean',
    'AVG_DOWNHOLE_PRESSURE':    'mean',
    'WCT':          'mean',
    'GOR' :         'median',
    'DRAWDOWN':     'mean',
    'DAYS_ON_PROD' : 'max',
    'ON_STREAM_HRS' : 'sum',
}

monthly_well_data = {}

for well_name,well_df in df.groupby('NPD_WELL_BORE_NAME'):
    if well_name not in TARGET_WELLS:
        print(f'Skipping wells : {well_name}')
        continue
        
    cols_available = { k: v for k, v in AGG_RULES.items() if k in well_df.columns}

    monthly = well_df[list(cols_available.keys())].resample('ME').agg(cols_available).dropna(
        subset=['OIL_RATE_NORM']
    )

    monthly_well_data[well_name] = monthly
    print(f'{well_name} : {len(monthly)} months')

#   Feature engineering 

##  Lag_Feature :  
#   what was producction value last month, 3 months, 6 months   Was not used in ARPS as is was a mathematical equation only

##  Rolling features : 
#   it smooths the value , means make average of last months production 
##  3month avg, 6 month avg  , it smooths the data as with average we remove the noisy data and give XG boost more stable data

##  Decline Rate : 
#   (present_month - Last ) / last
#3  this values helps XG boost learn in more better way as now we see change in each month

## WCT  :                   water cut , normalized (0 - 1)
## AVG_WHP_P :              well head pressure.
## DAYS_ON_PROD :           age of well
## AVG_DOWNHOLE_PRESSURE :  reserviour energy
## DRAWDOWN:                pressure diff as it is driving the flow


## monthly dataframe for each well seperatly then add feature engineering 
def engineer_features(month_df):
    df_feature = month_df.copy()


    ## lag features 
    df_feature['lag_1_oil'] = df_feature['OIL_RATE_NORM'].shift(1)   ## we get value for prev means shift col down by 1 row
    df_feature['lag_3_oil'] = df_feature['OIL_RATE_NORM'].shift(3)
    df_feature['lag_6_oil'] = df_feature['OIL_RATE_NORM'].shift(6)


    df_feature['lag_1_wct'] = df_feature['WCT'].shift(1) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_3_wct'] = df_feature['WCT'].shift(3) if 'WCT' in df_feature.columns else np.nan
    df_feature['lag_6_wct'] = df_feature['WCT'].shift(6) if 'WCT' in df_feature.columns else np.nan

    df_feature['lag_1_whp'] = df_feature['AVG_WHP_P'].shift(1)  if 'AVG_WHP_P' in df_feature.columns else np.nan

    ## rolling average
    ## rolling takes window of 3months and takes average of that
    df_feature['rolling_3m'] = df_feature['OIL_RATE_NORM'].rolling(window = 3,min_periods =2).mean()
    df_feature['rolling_6m'] = df_feature['OIL_RATE_NORM'].rolling(window = 6,min_periods =3).mean()

    ## month over percentage change
    ## pct_change()=   is (current - prev)/prev
    df_feature['month_over_month'] = df_feature['OIL_RATE_NORM'].pct_change().clip(-1,1) ## clip removes extreme

    ## rel performance
    df_feature['rate_vs_6m_avg'] = df_feature['lag_1_oil'] / df_feature['rolling_6m']

    ## Water cut change
    ## inc in this means water prod increases so it reduces oil prod
    df_feature['wct_change'] = df_feature['WCT'].diff()

    df_feature['well_age'] = df_feature['DAYS_ON_PROD'] / 365


    return df_feature



print(f'\n \tWalk Forward Cross validation')

FEATURE_COLS = [
    'lag_1_oil',              # last month production
    'lag_3_oil',              # 3 months ago production
    'lag_6_oil',              # 6 months ago production
    'lag_1_wct',              # last month water cut
    'lag_3_wct',              # last month water cut
    'lag_6_wct',              # last month water cut
    'lag_1_whp',              # last month wellhead pressure
    'rolling_3m',             # 3-month rolling average
    'rolling_6m',             # 6-month rolling average
    'month_over_month',       # recent decline rate
    'rate_vs_6m_avg',         # current vs historical average
    'WCT',                    # water cut (domain feature)
    'AVG_WHP_P',              # wellhead pressure (domain feature)
    'DAYS_ON_PROD',           # well age (domain feature)
    'AVG_DOWNHOLE_PRESSURE',  # reservoir energy (domain feature)
    'DRAWDOWN',               # pressure differential (domain feature)
    'GOR',                    # gas-oil ratio (domain feature)
    'wct_change',             # how fast water cut is rising
    'well_age',
]

Target_col = 'OIL_RATE_NORM'
xgb_result = {}
MIN_TRAIN_MONTH =18
xgb_result = {}

for well_name in TARGET_WELLS:
    if well_name not in TARGET_WELLS:
        print(f'Skipping wells : {well_name}')
        continue 
    monthly = monthly_well_data[well_name]
    df_feat= engineer_features(monthly)
    print(df_feat.head(10))

    df_feat = df_feat.dropna(subset=[Target_col])

    peak_idx= df_feat[Target_col].idxmax()
    peak_pos = df_feat.index.get_loc(peak_idx)
    decline_series = df_feat.iloc[peak_pos:].copy()

    n_total = len(decline_series)
    n_train = int(n_total * 0.8)
    n_test = n_total - n_train

    train_df = decline_series.iloc[:n_train]
    test_df = decline_series.iloc[n_train:]
    print(f'Decline month: {n_total} | Train : {n_train} | Test: {n_test}')
    print(f' Train dates {train_df.index[0].date()} --> {train_df.index[-1].date()}')
    print(f' Test dates {test_df.index[0].date()} --> {test_df.index[-1].date()}')

    wf_predictions = []
    wf_actuals =[]
    for i in range(MIN_TRAIN_MONTH , len(train_df)):
        ## train  till position i
        wf_train = train_df.iloc[:i]

        feature_avail = [c for c in FEATURE_COLS if c in wf_train.columns]

        wf_train_clean = wf_train[feature_avail + [Target_col]].dropna()

        if(len(wf_train_clean) <10):
            print(f'less Data to train on')
            continue
        x_wf = wf_train_clean[feature_avail]
        y_wf = wf_train_clean[Target_col]

        next_row = train_df.iloc[[i]][feature_avail]
        if next_row.isnull().any().any():       ## first any means is there any null value in column so basically it is series and then other any is if there is any column that has null value
                                                ## so is null gives u true or false if any col is null so first any gives bool in series then,other checks if any true value in series 
            continue

        model_wf = xgb.XGBRegressor(
            n_estimators = 100,
            max_depth =3,
            learning_rate = 0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha = 0.1,
            random_state = 42,
            verbosity =0,
        )
        model_wf.fit(x_wf,y_wf)
        pred = model_wf.predict(next_row)[0]   # predict could return array even if one value so we need to access index 0
        ##3 add predicted value and actual value what it was
        wf_predictions.append(pred)
        wf_actuals.append(train_df.iloc[i][Target_col])

    if(len(wf_actuals) > 0):
            wf_rmse = np.sqrt(mean_squared_error(wf_actuals,wf_predictions))
            wf_r2 = r2_score(wf_actuals,wf_predictions)
            wf_mae = mean_absolute_error(wf_actuals,wf_predictions)
            print(f'Walk Forward Cross validation RMSE: {wf_rmse}, R2={wf_r2} MAE:{wf_mae}')
    else:
            print(f'Walk forward :not enough data')


    feat_available = [c for c in FEATURE_COLS if c in train_df.columns]
    train_clean = train_df[feat_available +[Target_col]].dropna()
    test_clean = test_df[feat_available +[Target_col]].dropna()

    print(f'Final model :{len(train_clean)} training rows\n{len(test_clean)} clean tets rows')

    if len(train_clean) < 10 or len(test_clean) <3:
         print(f'Insufficient clean data {well_name}')
         continue
    
    X_train = train_clean[feat_available]
    Y_train = train_clean[Target_col]
    X_test = test_clean[feat_available]
    Y_test = test_clean[Target_col]

    final_model = xgb.XGBRegressor(
            n_estimators = 100,
            max_depth =3,
            learning_rate = 0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha = 0.1,
            random_state = 42,
            verbosity =0,
    )

    final_model.fit(X_train,Y_train)

    y_pred_test = final_model.predict(X_test)

    test_rmse = np.sqrt(mean_squared_error(Y_test,y_pred_test))
    test_mae = mean_absolute_error(Y_test,y_pred_test)
    test_r2 = r2_score(Y_test,y_pred_test)

    print(f'Test RMSE={test_rmse:.1f}  MAE={test_mae:.1f}  R²={test_r2:.3f}')
    print(f'Saving model for well : {well_name}')
    model_path = f'{MODELS_DIR}/xgb_{well_name.replace("/","-").replace(" ","_")}.joblib'
    joblib.dump(final_model, model_path)
    xgb_result[well_name] = {
        'train_df':       train_df,
        'test_df':        test_clean,
        'feat_available': feat_available,
        'X_train':        X_train,
        'Y_train':        Y_train,
        'X_test':         X_test,
        'Y_test':         Y_test,
        'y_pred_test':    y_pred_test,
        'model':          final_model,
        'wf_rmse':        wf_rmse,
        'test_rmse':      test_rmse,
        'test_mae':       test_mae,
        'test_r2':        test_r2,
        'decline_series':     decline_series,
        'n_train':        n_train,
    }
    

	 Loading data 
Loaded  8,020 rows
Resampling to monthly
Skipping wells : 15/9-F-1 C
Skipping wells : 15/9-F-11
15/9-F-12 : 102 months
15/9-F-14 : 97 months
Skipping wells : 15/9-F-15 D
Skipping wells : 15/9-F-5

 	Walk Forward Cross validation
            OIL_RATE_NORM   AVG_WHP_P  AVG_DOWNHOLE_PRESSURE       WCT  \
DATEPRD                                                                  
2008-02-29    2833.566248  112.894222             293.861056  0.032428   
2008-03-31    3008.567528  102.518300             278.453133  0.000450   
2008-04-30    2894.036903   93.011517             264.300414  0.005911   
2008-05-31    4207.003616   85.575233             262.864333  0.042744   
2008-06-30    4933.121906   74.945667             258.793533  0.003144   
2008-07-31    3877.515876   89.050194             264.462806  0.002514   
2008-08-31    3420.765141   88.166433             261.117967  0.003244   
2008-09-30    3193.573699   83.503269             252.014923  0.002172   
2008-10-31    3

NameError: name 'xbg_result' is not defined